# JamShield pipeline (Reviewer E)

1. **Label**: binary attack/benign from the `attack` column (constant per file).
2. **LOS/NLOS** is descriptive only: 5 of 20 files carry the tag, and only in the file name. It is not used as a split regime.
3. **Protocol**: identical to the Kosmanos pipeline, a 60/20/20 split by pseudo-episode (`split_by_episode` from `pcmef_fixed.py`) over all 20 files.
4. **Models**: RF-Stat and LSTM with the "modified" architecture (as cell D on Kosmanos: 2-layer BiLSTM 128→64 + dense head; 15 s windows with 7 statistics per feature for RF-Stat). BERT is not used.
5. **Seeds**: the same 20 seeds as the Kosmanos experiments.

Dependencies (copy to the Drive paths used below): `pcmef_fixed.py`, `pcmef_v2_infra.py` (sanity checks), `jamshield_e59_counter.py` (counter-column fix).

## Part 1: Data loading

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Import `pcmef_fixed.py`

In [ ]:
import shutil, sys, importlib, os

PCMEF_FIXED_PATH = '/content/drive/MyDrive/Detector_Results/pcmef_fixed.py'

shutil.copy(PCMEF_FIXED_PATH, '/content/pcmef_fixed.py')
sys.path.insert(0, '/content')
importlib.invalidate_caches()

import pcmef_fixed as pf
print("pcmef_fixed berhasil diimport. Jumlah SEEDS:", len(pf.SEEDS))

### 1.3 Locate the JamShield dataset

In [ ]:
DRIVE_JAMSHIELD_PATH = '/content/drive/MyDrive/JamShield_Dataset'

In [ ]:
import zipfile

EXTRACT_DIR = '/content/jamshield_data'

def resolve_data_dir(path, extract_dir=EXTRACT_DIR):
    """Terima path folder ATAU file .zip, kembalikan folder yang isinya 20 CSV."""
    if os.path.isdir(path):
        csvs = [f for f in os.listdir(path) if f.endswith('.csv')]
        if len(csvs) == 20:
            return path
        for root, dirs, files in os.walk(path):
            csvs = [f for f in files if f.endswith('.csv')]
            if len(csvs) == 20:
                return root
        raise FileNotFoundError(
            f"Folder '{path}' ada, tapi tidak ketemu 20 file CSV di dalamnya "
            f"(ketemu {len(csvs)}). Cek lagi isi foldernya."
        )
    elif os.path.isfile(path) and path.endswith('.zip'):
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path, 'r') as z:
            z.extractall(extract_dir)
        for root, dirs, files in os.walk(extract_dir):
            csvs = [f for f in files if f.endswith('.csv')]
            if len(csvs) == 20:
                return root
        raise FileNotFoundError(
            f"Sudah di-extract ke '{extract_dir}', tapi tidak ketemu folder "
            f"dengan 20 file CSV. Cek struktur zip-nya."
        )
    else:
        raise FileNotFoundError(f"Path '{path}' bukan folder atau file .zip yang valid.")

DATA_DIR = resolve_data_dir(DRIVE_JAMSHIELD_PATH)
print(f"DATA_DIR ditemukan: {DATA_DIR}")
print(f"Isi: {sorted(os.listdir(DATA_DIR))}")

### 1.4 Loading functions (self-contained)

In [ ]:
import glob
import pandas as pd
import numpy as np

RAW_FEATURE_COLS = [
    'tx_total_pkts', 'tx_total_bytes', 'tx_ucast_pkts', 'tx_ucast_bytes',
    'tx_mcast_bcast_pkts', 'tx_mcast_bcast_bytes', 'tx_failures',
    'rx_data_pkts', 'rx_data_bytes', 'rx_ucast_pkts', 'rx_ucast_bytes',
    'rx_mcast_bcast_pkts', 'rx_mcast_bcast_bytes', 'rx_decrypt_succeeds',
    'tx_data_pkts_retried', 'tx_total_pkts_sent', 'tx_pkts_retries',
    'tx_pkts_retry_exhausted', 'rx_total_pkts_retried',
    'rate_last_tx_pkt_min', 'rate_last_tx_pkt_max',
    'per_antenna_rssi_last_rx_data_frame_1', 'per_antenna_rssi_last_rx_data_frame_2',
    'per_antenna_rssi_last_rx_data_frame_3', 'per_antenna_rssi_last_rx_data_frame_4',
    'per_antenna_avg_rssi_rx_data_frames_1', 'per_antenna_avg_rssi_rx_data_frames_2',
    'per_antenna_avg_rssi_rx_data_frames_3', 'per_antenna_avg_rssi_rx_data_frames_4',
    'per_antenna_noise_floor_1', 'per_antenna_noise_floor_2',
    'per_antenna_noise_floor_3', 'per_antenna_noise_floor_4',
    'sinr_per_antenna_1', 'sinr_per_antenna_2', 'sinr_per_antenna_3', 'sinr_per_antenna_4',
]  # 37 kolom fitur mentah (tanpa sample, station, attack)

SAMPLING_INTERVAL_SEC = 0.5   # tetap, sesuai dokumentasi dataset JamShield
EPISODE_BUCKET_SEC = 60       # SAMA seperti RF Jamming (floor(Time/60))


def parse_filename(fname):
    """Ekstrak jammer_type, variant, dan tag LOS/NLOS (jika ada) dari NAMA file."""
    stem = os.path.basename(fname).replace('.csv', '')

    if stem.startswith('data_benign'):
        jammer_type, variant = 'benign', stem.replace('data_benign_', 'benign_')
    elif stem.startswith('constant_jammer'):
        jammer_type = 'constant'
        variant = stem.replace('constant_jammer_', '')
    elif stem.startswith('random_jammer'):
        jammer_type = 'random'
        variant = stem.replace('random_jammer_', '')
    elif stem.startswith('reactive_jammer'):
        jammer_type = 'reactive'
        variant = stem.replace('reactive_jammer_', '')
    else:
        jammer_type, variant = 'unknown', stem

    if stem.endswith('_LOS'):
        los_nlos = 'LOS'
    elif stem.endswith('_NLOS'):
        los_nlos = 'NLOS'
    else:
        los_nlos = None  # 15/20 file: memang tidak ada tag -> BUKAN bug
    return jammer_type, variant, los_nlos


def load_jamshield(data_dir, verbose=True):
    """Load seluruh 20 file JamShield -> satu DataFrame."""
    files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
    assert len(files) == 20, f"Ekspektasi 20 file, ketemu {len(files)}. Cek DATA_DIR."

    frames = []
    for f in files:
        df = pd.read_csv(f)
        jammer_type, variant, los_nlos = parse_filename(f)

        uniq_attack = df['attack'].unique()
        assert len(uniq_attack) == 1, f"{f}: kolom attack TIDAK konstan! nilai={uniq_attack}"
        y_val = int(uniq_attack[0])

        df = df.sort_values('sample').reset_index(drop=True)
        df['station_seq'] = df.groupby('station').cumcount()
        df['time_sec'] = df['station_seq'] * SAMPLING_INTERVAL_SEC

        df['y'] = y_val
        df['source_file'] = os.path.basename(f)
        df['jammer_type'] = jammer_type
        df['variant'] = variant
        df['los_nlos'] = los_nlos

        bucket = (df['time_sec'] // EPISODE_BUCKET_SEC).astype(int)
        df['pseudo_episode_id'] = (
            df['source_file'] + '__' + df['station'] + '__' + bucket.astype(str)
        )

        frames.append(df)

    full = pd.concat(frames, ignore_index=True)

    if verbose:
        print(f"Total baris            : {len(full)}")
        print(f"Total file             : {full['source_file'].nunique()}")
        print(f"Distribusi label y     : {full['y'].value_counts().to_dict()}")
        print(f"Jumlah pseudo-episode  : {full['pseudo_episode_id'].nunique()}")
        print("Episode per kelas      :")
        print(full.groupby('y')['pseudo_episode_id'].nunique().to_string())
        ep_len = full.groupby('pseudo_episode_id').size()
        print(f"Panjang episode (baris): min={ep_len.min()} median={ep_len.median()} max={ep_len.max()}")
        print("\nCakupan LOS/NLOS (SEKUNDER -- hanya utk dilaporkan, bukan split utama):")
        print(full.groupby(full['los_nlos'].fillna('NoTag'))['source_file'].nunique().to_string())

    return full

print("Fungsi Bagian 1 siap dipakai.")

### 1.5 Run loading

In [ ]:
df_loaded = load_jamshield(DATA_DIR)
df_loaded.head()

### 1.6 Save checkpoint

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/JamShield_Dataset/jamshield_loaded.parquet'
try:
    df_loaded.to_parquet(CHECKPOINT_PATH, index=False)
except ImportError:
    # fallback if pyarrow/fastparquet is not available in this Colab runtime
    !pip install -q pyarrow
    df_loaded.to_parquet(CHECKPOINT_PATH, index=False)
print(f"Tersimpan: {CHECKPOINT_PATH}  ({os.path.getsize(CHECKPOINT_PATH) / 1e6:.2f} MB)")

### E5.8: File inventory (label, station, duration, episodes, LOS/NLOS tag)
Confirms 4 benign and 16 attack files. Reads `df_loaded` only; no training.

In [ ]:
inv = []
for f, g in df_loaded.groupby('source_file'):
    jt, variant, los = parse_filename(f)
    stn = [c for c in ('station','Station') if c in g.columns]
    inv.append(dict(file=f, jammer=jt, label=int(g['y'].mode().iloc[0]),
                    n_baris=len(g),
                    n_station=(g[stn[0]].nunique() if stn else None),
                    durasi_s=float(g['time_sec'].max() - g['time_sec'].min()),
                    n_episode=g['pseudo_episode_id'].nunique(),
                    LOS_NLOS=los or '-'))
inv = pd.DataFrame(inv).sort_values(['label','jammer','file']).reset_index(drop=True)
display(inv)
n_ben, n_att = int((inv.label==0).sum()), int((inv.label==1).sum())
print(f'Benign={n_ben} | Attack={n_att} | total={len(inv)}  (harus 4 / 16 / 20)')
assert (n_ben, n_att) == (4, 16), 'Inventaris tidak sesuai ekspektasi 4 benign + 16 attack -> cek label y'
print('Tag LOS/NLOS:', inv.LOS_NLOS.value_counts().to_dict(), '(harus 5 bertag, 15 tanpa tag)')
inv.to_csv('/content/drive/MyDrive/JamShield_Dataset/E5_8_inventaris_file.csv', index=False)

In [ ]:
import pandas as pd

print("=== Cek struktur source_file di JamShield ===")
print(f"Jumlah source_file unik: {df_loaded['source_file'].nunique()}")
print(f"Jumlah episode unik (pseudo_episode_id): {df_loaded['pseudo_episode_id'].nunique() if 'pseudo_episode_id' in df_loaded.columns else 'kolom tidak ditemukan'}")

# cek apakah tiap source_file punya 1 label konsisten atau campur
label_col = 'y' if 'y' in df_loaded.columns else 'label'
per_file_label_count = df_loaded.groupby('source_file')[label_col].nunique()
n_mixed = (per_file_label_count > 1).sum()
n_total_files = per_file_label_count.shape[0]
print(f"\nSource_file dengan >1 label berbeda: {n_mixed}/{n_total_files}")

if n_mixed > 0:
    print("\nContoh file campur label (5 teratas):")
    mixed_files = per_file_label_count[per_file_label_count > 1].index[:5]
    for f in mixed_files:
        sub = df_loaded[df_loaded['source_file'] == f]
        print(f"  {f}: label distribusi = {sub[label_col].value_counts().to_dict()}")

# distribusi label mayoritas per file (untuk keperluan stratifikasi nanti)
majority_label_per_file = df_loaded.groupby('source_file')[label_col].agg(lambda x: x.value_counts().idxmax())
print(f"\nDistribusi label MAYORITAS per file (dipakai untuk stratifikasi StratifiedGroupKFold):")
print(majority_label_per_file.value_counts())

# cek juga di level episode: apakah satu pseudo_episode_id selalu berasal dari satu source_file yang sama
if 'pseudo_episode_id' in df_loaded.columns:
    ep_file_count = df_loaded.groupby('pseudo_episode_id')['source_file'].nunique()
    n_ep_mixed_file = (ep_file_count > 1).sum()
    print(f"\nEpisode yang berasal dari >1 source_file (seharusnya 0): {n_ep_mixed_file}")

In [ ]:
# One-off: archive older E5.9 result JSONs so that the counter-fix run is recomputed and each result file carries
# `balanced_accuracy`. Leave RUN_BACKUP = False on normal runs: re-running this cell after the E5.9 cell would archive the new results too.
import shutil, glob, os
RUN_BACKUP = False
if RUN_BACKUP:
    src_dir = '/content/drive/MyDrive/JamShield_Dataset/results_e59'
    bak = src_dir + '/backup_before_balanced_accuracy'
    os.makedirs(bak, exist_ok=True)
    for f in glob.glob(f'{src_dir}/*.json'):
        shutil.move(f, bak)
    print('archived:', len(glob.glob(f'{bak}/*.json')), 'files (expected 20)')
else:
    print('RUN_BACKUP = False: existing E5.9 results are reused.')

In [ ]:
import shutil, os, json, importlib
import numpy as np

shutil.copy('/content/drive/MyDrive/JamShield_Dataset/jamshield_e59_counter.py',
            '/content/jamshield_e59_counter.py')
import jamshield_e59_counter as e59
importlib.reload(e59)

OUT_E59 = '/content/drive/MyDrive/JamShield_Dataset/results_e59'
os.makedirs(OUT_E59, exist_ok=True)

# Langkah 1-2: identifikasi kolom counter + ubah jadi diff (pada df_loaded MENTAH)
counter_cols, pct_table = e59.identify_counter_columns(df_loaded, RAW_FEATURE_COLS)
print(f"Kolom counter teridentifikasi: {len(counter_cols)}/{len(RAW_FEATURE_COLS)}")

df_fixed = e59.apply_counter_diff(df_loaded, counter_cols)
df_fixed = df_fixed.rename(columns={
    'pseudo_episode_id': 'pseudo_run_id', 'time_sec': 'Time', 'y': 'label',
})

# Restrict to episodes >= 15 s (same population as the main pipeline)
df_fixed, feasible_ids, n_dropped = pf.filter_feasible_episodes(df_fixed, min_duration_sec=15)
print(f"Episode dibuang (durasi < 15s): {n_dropped}")

pf.FEATURES = list(RAW_FEATURE_COLS)

results_e59 = []
for seed in pf.SEEDS:
    fp = os.path.join(OUT_E59, f'JamShield_RFStat_counterfix_seed{seed}.json')
    if os.path.exists(fp):
        results_e59.append(json.load(open(fp)))
        continue
    r = e59.run_rf_jamshield_fixed(df_fixed, seed, len(RAW_FEATURE_COLS))
    if r is None:
        print(f'  [seed {seed}] dilewati (data tidak cukup)')
        continue
    json.dump(r, open(fp, 'w'), indent=2)
    results_e59.append(r)
    print(f'  [seed {seed:4d}] acc={r["accuracy"]:.4f} f1={r["f1_macro"]:.4f}')

accs = np.array([r['accuracy'] for r in results_e59])
OLD_MEAN_ACC = 0.9929  # RF-Stat before the counter fix and before the >= 15 s filter
print(f"\n=== RESULT AFTER COUNTER FIX + >= 15 s FILTER (n={len(accs)} seed) ===")
print(f"  Accuracy mean = {accs.mean()*100:.2f}%  (SD={accs.std(ddof=1)*100:.2f}%)")
print(f"  Accuracy LAMA (baseline awal, 20 seed) = {OLD_MEAN_ACC*100:.2f}%")
print(f"  Selisih = {(accs.mean()-OLD_MEAN_ACC)*100:+.2f} poin persentase")

In [ ]:
b = np.array([r['balanced_accuracy'] for r in results_e59]) * 100
print(f'E5.9 balanced accuracy = {b.mean():.2f} ± {b.std(ddof=1):.2f}   (n={len(b)} seeds)')

## Protocol split, RF-Stat/LSTM (20 seeds), statistical aggregation

Core functions (`split_by_episode`, `create_windows_keyed`, `extract_tabular_features`, `pad_list`, `build_lstm`, `compute_metrics`, `bootstrap_ci`) are reused from `pcmef_fixed.py`.

### E5.10: file-level evaluation (LOFO pooled and StratifiedGroupKFold k=4 × 20 seeds)

In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score

# ======================================================================
# E5.10: LOFO pooled + StratifiedGroupKFold k=4 x 20 seeds
# Hyperparameters fixed to the RF-Stat cell D configuration (run_rf_jamshield).
# ======================================================================

WINDOW_SEC, STEP_SEC = 15, 1
RF_PARAMS = dict(n_estimators=200, max_depth=15, min_samples_split=5,
                  min_samples_leaf=2, class_weight='balanced', n_jobs=-1)

print("=== Precompute window features sekali utk seluruh df_fixed ===")
Xall_l, yall, kall = pf.create_windows_keyed(df_fixed, WINDOW_SEC, STEP_SEC)
Xall = pf.extract_tabular_features(Xall_l)
yall = np.array(yall)
episode_ids_w = np.array([k[0] for k in kall])

episode_to_file = df_fixed.groupby('pseudo_run_id')['source_file'].first().to_dict()
file_of_window = np.array([episode_to_file[e] for e in episode_ids_w])
print(f"Total window: {len(yall)}  |  file unik: {len(set(file_of_window))}")

def fit_eval(Xtr, ytr, Xte, seed):
    sc = StandardScaler().fit(Xtr)
    rf = RandomForestClassifier(random_state=seed, **RF_PARAMS)
    rf.fit(sc.transform(Xtr), ytr)
    return rf.predict(sc.transform(Xte))

OUT_E510 = '/content/drive/MyDrive/JamShield_Dataset/results_e510_v2'
os.makedirs(OUT_E510, exist_ok=True)

# --- 1. Leave-one-file-out, prediksi pooled ---
fp_lofo = os.path.join(OUT_E510, 'LOFO_pooled.json')
if os.path.exists(fp_lofo):
    r_lofo = json.load(open(fp_lofo))
else:
    files_all = sorted(set(file_of_window))
    yt_pool, yp_pool = [], []
    for held_out in files_all:
        te_mask = file_of_window == held_out
        tr_mask = ~te_mask
        Xtr, ytr = Xall[tr_mask], yall[tr_mask]
        Xte, yte = Xall[te_mask], yall[te_mask]
        if len(np.unique(ytr)) < 2 or len(Xte) == 0:
            print(f'  [LOFO] lewati {held_out} (kelas train tidak lengkap / test kosong)')
            continue
        pred = fit_eval(Xtr, ytr, Xte, seed=42)
        yt_pool.append(yte); yp_pool.append(pred)
        print(f'  [LOFO] held_out={held_out}: n_test={len(yte)}')
    yt_pool = np.concatenate(yt_pool); yp_pool = np.concatenate(yp_pool)
    r_lofo = dict(
        balanced_accuracy=float(balanced_accuracy_score(yt_pool, yp_pool)),
        f1_macro=float(f1_score(yt_pool, yp_pool, average='macro')),
        n_pooled=int(len(yt_pool)),
    )
    json.dump(r_lofo, open(fp_lofo, 'w'), indent=2)

print(f"\n=== LOFO pooled: balanced_acc={r_lofo['balanced_accuracy']*100:.2f}%  "
      f"f1_macro={r_lofo['f1_macro']:.4f}  (n_pooled={r_lofo['n_pooled']}) ===")

# --- 2. StratifiedGroupKFold k=4, diulang 20 seed ---
sgkf_results = []
for seed in pf.SEEDS:
    fp = os.path.join(OUT_E510, f'SGKF_k4_seed{seed}.json')
    if os.path.exists(fp):
        sgkf_results.append(json.load(open(fp))); continue
    sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    yt_pool, yp_pool = [], []
    for fold_i, (tr_idx, te_idx) in enumerate(sgkf.split(Xall, yall, groups=file_of_window)):
        Xtr, ytr = Xall[tr_idx], yall[tr_idx]
        Xte, yte = Xall[te_idx], yall[te_idx]
        if len(np.unique(ytr)) < 2 or len(Xte) == 0:
            print(f'  [seed {seed} fold {fold_i}] dilewati (kelas tdk lengkap)'); continue
        pred = fit_eval(Xtr, ytr, Xte, seed=seed)
        yt_pool.append(yte); yp_pool.append(pred)
    if not yt_pool:
        print(f'  [seed {seed}] GAGAL total, dilewati'); continue
    yt_pool = np.concatenate(yt_pool); yp_pool = np.concatenate(yp_pool)
    r = dict(seed=int(seed),
             balanced_accuracy=float(balanced_accuracy_score(yt_pool, yp_pool)),
             f1_macro=float(f1_score(yt_pool, yp_pool, average='macro')),
             n_pooled=int(len(yt_pool)))
    json.dump(r, open(fp, 'w'), indent=2)
    sgkf_results.append(r)
    print(f'  [seed {seed:4d}] balanced_acc={r["balanced_accuracy"]*100:.2f}%  f1_macro={r["f1_macro"]:.4f}')

bal_accs = np.array([r['balanced_accuracy'] for r in sgkf_results])
f1s = np.array([r['f1_macro'] for r in sgkf_results])
print(f"\n=== StratifiedGroupKFold k=4 (n={len(bal_accs)} seed) ===")
print(f"  balanced_accuracy: mean={bal_accs.mean()*100:.2f}%  SD={bal_accs.std(ddof=1)*100:.2f}%")
print(f"  f1_macro:          mean={f1s.mean():.4f}  SD={f1s.std(ddof=1):.4f}")

# --- 3. Baseline kelas mayoritas (dihitung pada pooled label LOFO, utk referensi) ---
majority_class = int(pd.Series(yall).mode()[0])
yp_majority = np.full_like(yall, majority_class)
bal_acc_majority = balanced_accuracy_score(yall, yp_majority)
f1_majority = f1_score(yall, yp_majority, average='macro')
print(f"\n=== Baseline kelas mayoritas (prediksi semua = kelas {majority_class}) ===")
print(f"  balanced_accuracy={bal_acc_majority*100:.2f}%  f1_macro={f1_majority:.4f}")
print(f"  (balanced accuracy 50% adalah nilai teoretis utk klasifikasi biner tanpa diskriminasi sama sekali)")

### E5.10b: per-file accuracy under leave-one-file-out (Section 4.7, Reviewer E)

Repeats the LOFO of the cell above with exactly the same settings (`fit_eval`, `RF_PARAMS`, seed 42) but keeps the per-window predictions, so accuracy can be reported per recording. File labels are constant, so per-file accuracy is the false-positive rate complement for benign files (1 − FPR) and the detection rate for attack files. **Check first:** the printed pooled values must equal the LOFO row of Table 8 (balanced accuracy 67.39, macro-F1 68.87); if not, stop, because the settings differ from E5.10. Requires `jamshield_per_file.py` in `Detector_Results/`. Takes about 15-30 min on CPU. Writes `LOFO_per_file.csv` and `LOFO_per_file.json` to `results_e510_v2/`.

In [ ]:
import shutil, sys
shutil.copy('/content/drive/MyDrive/Detector_Results/jamshield_per_file.py', '/content/jamshield_per_file.py')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import jamshield_per_file as JPF
per_file = JPF.run(Xall, yall, file_of_window, fit_eval, OUT_E510)   # add benign_label=0 if file names lack 'benign'

### StratifiedGroupKFold anomaly at seed 777: bug, sklearn behaviour, or design?

Run after the E5.10 cell above (uses `Xall, yall, file_of_window, fit_eval, OUT_E510, RF_PARAMS`).

**Question:** a fold can end up with no benign file in train. Is that a bug in our code, or does sklearn's `StratifiedGroupKFold` simply not guarantee one benign file per fold? It balances the number of *samples*, not *files*, and `data_benign_1.csv` holds about 52% of the benign rows.

**Cells:** fold composition of seed 777 and input checks → size of the problem (20 seeds × 4 folds) → manual folds (one benign file per fold, attack files round-robin per jammer type) and SGKF re-run → documentation.

**Output:** fold composition per seed, number of degenerate folds, SGKF with manual folds vs sklearn folds. Pooled LOFO is not affected and is not repeated.

In [ ]:
# 5.1-5.2 + 5.4: komposisi fold StratifiedGroupKFold (sklearn) -- seed 777 dicetak rinci, lalu 20 seed x 4 fold
files_all = sorted(set(file_of_window))
file_label  = {f: int(pd.Series(yall[file_of_window == f]).mode()[0]) for f in files_all}   # 0=benign, 1=attack
file_jammer = {f: parse_filename(f)[0] for f in files_all}                                # benign/constant/random/reactive
print('5.2 input SGKF: groups = source_file (n=%d file) | y = label biner per WINDOW | shuffle=True, random_state=seed | k=4' % len(files_all))
assert len(file_of_window) == len(yall)

def fold_table(fold_of_window, seed, tag):
    rows = []
    for k in sorted(set(fold_of_window)):
        te = fold_of_window == k; tr = ~te
        f_te, f_tr = sorted(set(file_of_window[te])), sorted(set(file_of_window[tr]))
        rows.append(dict(tag=tag, seed=seed, fold=int(k),
                         ben_file_train=sum(file_label[f] == 0 for f in f_tr), ben_file_test=sum(file_label[f] == 0 for f in f_te),
                         att_file_train=sum(file_label[f] == 1 for f in f_tr), att_file_test=sum(file_label[f] == 1 for f in f_te),
                         win_ben_test=int((yall[te] == 0).sum()), win_att_test=int((yall[te] == 1).sum()),
                         win_ben_train=int((yall[tr] == 0).sum()), files_test=f_te))
    return pd.DataFrame(rows)

def sklearn_fold_of_window(seed, k=4):
    fw = np.full(len(yall), -1)
    for i, (_, te_idx) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(Xall, yall, groups=file_of_window)):
        fw[te_idx] = i
    return fw

t777 = fold_table(sklearn_fold_of_window(777), 777, 'sklearn')
print('\n=== 5.1 Seed 777, sklearn SGKF ==='); print(t777.drop(columns='files_test').to_string(index=False))
for _, r in t777.iterrows(): print(f"  fold {r.fold} test files: {[f.replace('.csv','') for f in r.files_test]}")

comp_sk = pd.concat([fold_table(sklearn_fold_of_window(s), s, 'sklearn') for s in pf.SEEDS], ignore_index=True)
comp_sk.drop(columns='files_test').to_csv(f'{OUT_E510}/E5_10_fold_composition_sklearn.csv', index=False)
print(f'\n=== 5.4 Skala masalah sklearn SGKF: {comp_sk.seed.nunique()} seed x 4 fold = {len(comp_sk)} fold ===')
print('  fold TANPA benign di train :', int((comp_sk.ben_file_train == 0).sum()))
print('  fold TANPA benign di test  :', int((comp_sk.ben_file_test == 0).sum()))
print('  fold dgn >1 file benign di test:', int((comp_sk.ben_file_test > 1).sum()))
print('  seed dgn >=1 fold degenerate  :', comp_sk[(comp_sk.ben_file_train == 0) | (comp_sk.ben_file_test == 0)].seed.nunique(), 'dari', comp_sk.seed.nunique())

In [ ]:
# 5.5-5.6: penugasan fold MANUAL (1 file benign per fold; attack dibagi round-robin per jenis jammer) + jalankan ulang SGKF 20 seed
def assign_folds(files, labels, jammer, seed, k=4):
    rng = np.random.RandomState(seed)
    benign = sorted(f for f in files if labels[f] == 0); attack = sorted(f for f in files if labels[f] == 1)
    rng.shuffle(benign)
    fold = {f: i % k for i, f in enumerate(benign)}                    # 1 benign per fold (kalau n_benign == k)
    c = 0
    for jt in sorted({jammer[f] for f in attack}):                     # round-robin per jenis jammer, hitungan menerus
        fs = sorted(f for f in attack if jammer[f] == jt); rng.shuffle(fs)
        for f in fs: fold[f] = c % k; c += 1
    return fold

OUT_E510_MAN = '/content/drive/MyDrive/JamShield_Dataset/results_e510_v3_manual'
os.makedirs(OUT_E510_MAN, exist_ok=True)
comp_man, man_results = [], []
for seed in pf.SEEDS:
    fold_map = assign_folds(files_all, file_label, file_jammer, seed)
    fw = np.array([fold_map[f] for f in file_of_window])
    comp_man.append(fold_table(fw, seed, 'manual'))
    fp = os.path.join(OUT_E510_MAN, f'SGKF_manual_k4_seed{seed}.json')
    if os.path.exists(fp): man_results.append(json.load(open(fp))); continue
    yt_pool, yp_pool = [], []
    for k_ in range(4):
        te_m = fw == k_; tr_m = ~te_m
        if len(np.unique(yall[tr_m])) < 2 or te_m.sum() == 0: print(f'  [seed {seed} fold {k_}] dilewati'); continue
        yp_pool.append(fit_eval(Xall[tr_m], yall[tr_m], Xall[te_m], seed=seed)); yt_pool.append(yall[te_m])
    yt_pool, yp_pool = np.concatenate(yt_pool), np.concatenate(yp_pool)
    r = dict(seed=int(seed), balanced_accuracy=float(balanced_accuracy_score(yt_pool, yp_pool)),
             f1_macro=float(f1_score(yt_pool, yp_pool, average='macro')), n_pooled=int(len(yt_pool)))
    json.dump(r, open(fp, 'w'), indent=2); man_results.append(r)
    print(f'  [manual seed {seed:4d}] balanced_acc={r["balanced_accuracy"]*100:.2f}%  f1_macro={r["f1_macro"]:.4f}')

comp_man = pd.concat(comp_man, ignore_index=True)
comp_man.drop(columns='files_test').to_csv(f'{OUT_E510}/E5_10_fold_composition_manual.csv', index=False)
print('\nKomposisi fold MANUAL: fold tanpa benign di train =', int((comp_man.ben_file_train == 0).sum()),
      '| tanpa benign di test =', int((comp_man.ben_file_test == 0).sum()), '| >1 benign di test =', int((comp_man.ben_file_test > 1).sum()))

sk_res = {r['seed']: r for r in (json.load(open(os.path.join(OUT_E510, f'SGKF_k4_seed{s}.json'))) for s in pf.SEEDS
                                 if os.path.exists(os.path.join(OUT_E510, f'SGKF_k4_seed{s}.json')))}
print('\n=== PERBANDINGAN SGKF k=4, 20 seed (RF-Stat, hiperparameter dikunci) ===')
for name, res in (('sklearn (versi lama)', list(sk_res.values())), ('fold manual (baru)', man_results)):
    b = np.array([r['balanced_accuracy'] for r in res]); f = np.array([r['f1_macro'] for r in res])
    print(f'  {name:22s} n={len(b)} | balanced_acc={b.mean()*100:.2f}% +- {b.std(ddof=1)*100:.2f}% | f1_macro={f.mean():.4f} +- {f.std(ddof=1):.4f}')
common = [s for s in pf.SEEDS if s in sk_res and any(r['seed'] == s for r in man_results)]
if common:
    d = np.array([next(r for r in man_results if r['seed'] == s)['balanced_accuracy'] - sk_res[s]['balanced_accuracy'] for s in common]) * 100
    rng_ = np.random.RandomState(0); bs = [rng_.choice(d, len(d)).mean() for _ in range(10000)]
    print(f'  selisih manual - sklearn (balanced_acc, per seed): {d.mean():+.2f} pp | bootstrap CI95 [{np.percentile(bs,2.5):+.2f}, {np.percentile(bs,97.5):+.2f}]')

In [ ]:
# 5.7 Dokumentasi E5.10 (dicetak supaya bisa disalin ke laporan/naskah)
print('Model       : RF-Stat saja (RandomForestClassifier) -- LSTM TIDAK dipakai di E5.10')
print('Set fitur   : %d fitur mentah JamShield, SETELAH counter-fix E5.9 (counter kumulatif -> diff), df_fixed, episodes >= 15 s' % len(pf.FEATURES))
print('Window      : 15 s, stride 1 s; fitur = pf.extract_tabular_features(window)')
print('Hiperparam  :', RF_PARAMS, '(dikunci = RF-Stat sel D); StandardScaler fit pada train fold')
print('Metrik      : balanced accuracy & macro-F1 pada prediksi POOLED lintas fold; 20 seed x k=4')
print('Evaluasi    : LOFO pooled (tidak terdampak masalah fold) + SGKF sklearn (lama) + SGKF fold manual (baru)')

In [ ]:
from sklearn.neighbors import NearestNeighbors

def audit_nn_distance_excl_same_episode(pf, tr_df, te_df, window_sec=15, step_sec=1,
                                          seed=42, k_search=50):
    """
    Sama seperti audit_nn_distance(), tapi saat mencari tetangga terdekat
    TRAIN->TRAIN, tetangga dari episode yang SAMA dengan query dikecualikan
    (bukan cuma exclude diri sendiri). k_search = berapa banyak kandidat
    tetangga awal yang dicari sebelum difilter episode-nya.
    """
    Xtr_l, ytr, ktr = pf.create_windows_keyed(tr_df, window_sec, step_sec)
    Xte_l, yte, kte = pf.create_windows_keyed(te_df, window_sec, step_sec)
    if not Xtr_l or not Xte_l:
        return None

    Xtr = pf.extract_tabular_features(Xtr_l)
    Xte = pf.extract_tabular_features(Xte_l)
    ep_tr = np.array([k[0] for k in ktr])   # episode id tiap window train

    sc = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr)
    Xte_s = sc.transform(Xte)

    # test -> train (episode test & train sudah pasti beda krn split_by_episode,
    # jadi tidak perlu filter di sini)
    nn_te = NearestNeighbors(n_neighbors=1).fit(Xtr_s)
    d_te_tr, _ = nn_te.kneighbors(Xte_s)
    d_te_tr = d_te_tr.ravel()

    # train -> train, EXCLUDE tetangga dari episode yang sama
    k_eff = min(k_search, len(Xtr_s))
    nn_tr = NearestNeighbors(n_neighbors=k_eff).fit(Xtr_s)
    dists, idxs = nn_tr.kneighbors(Xtr_s)   # shape (n_train, k_eff)

    d_tr_tr_excl = np.full(len(Xtr_s), np.nan)
    n_gagal = 0
    for i in range(len(Xtr_s)):
        for j in range(1, k_eff):  # j=0 selalu diri sendiri (dist=0), skip
            cand_idx = idxs[i, j]
            if ep_tr[cand_idx] != ep_tr[i]:
                d_tr_tr_excl[i] = dists[i, j]
                break
        else:
            n_gagal += 1  # k_search kurang besar, semua kandidat 1 episode yg sama

    if n_gagal > 0:
        print(f'  [peringatan] {n_gagal} window train gagal dapat tetangga beda-episode '
              f'dalam top-{k_search}, coba naikkan k_search')
    d_tr_tr_excl = d_tr_tr_excl[~np.isnan(d_tr_tr_excl)]

    p05_train = float(np.percentile(d_tr_tr_excl, 5))
    pct_test_closer = float(100.0 * np.mean(d_te_tr < p05_train))

    return dict(
        n_train_windows=len(Xtr), n_test_windows=len(Xte),
        n_gagal_exclude=int(n_gagal),
        d_test_to_train_mean=float(d_te_tr.mean()),
        d_train_to_train_excl_mean=float(d_tr_tr_excl.mean()),
        d_train_to_train_excl_median=float(np.median(d_tr_tr_excl)),
        p05_train_to_train_excl=p05_train,
        pct_test_closer_than_p05_train=pct_test_closer,
    )

OUT_E511 = '/content/drive/MyDrive/JamShield_Dataset/results_e511_v2'
os.makedirs(OUT_E511, exist_ok=True)
results_e511v2 = []
for seed in pf.SEEDS:
    fp = os.path.join(OUT_E511, f'JamShield_A5b_exclepisode_seed{seed}.json')
    if os.path.exists(fp):
        results_e511v2.append(json.load(open(fp))); continue
    tr_df, va_df, te_df = pf.split_by_episode(df_fixed, seed=seed)
    r = audit_nn_distance_excl_same_episode(pf, tr_df, te_df, seed=seed)
    if r is None:
        print(f'  [seed {seed}] dilewati'); continue
    r['seed'] = int(seed)
    json.dump(r, open(fp, 'w'), indent=2)
    results_e511v2.append(r)
    print(f'  [seed {seed:4d}] pct_test_closer_than_p05 = {r["pct_test_closer_than_p05_train"]:.2f}%  '
          f'(p05_excl={r["p05_train_to_train_excl"]:.4f}, gagal_exclude={r["n_gagal_exclude"]})')

pct_vals2 = np.array([r['pct_test_closer_than_p05_train'] for r in results_e511v2])
print(f"\n=== SUMMARY E5.11 (same-episode neighbours excluded, n={len(pct_vals2)} seed) ===")
print(f"  pct_test_closer_than_p05_train: mean={pct_vals2.mean():.2f}%  SD={pct_vals2.std(ddof=1):.2f}%  "
      f"min={pct_vals2.min():.2f}%  max={pct_vals2.max():.2f}%")

### Prepare dataframe (rename columns, override feature list)

In [ ]:
# ==========================================================================
# Functions for parts 2-4 (from jamshield_pipeline_part2_3.py). JamShield caches use
# per-model versioning (pf.cache_is_valid / pf.model_code_hash), so editing
# pcmef_fixed.py does not discard results that are still valid.
# Run this cell once before preparing the dataframe.
# ==========================================================================
import os
import json
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# pf = pcmef_fixed.py (imported in part 1.2)
import pcmef_fixed as pf


def prepare_jamshield_df(df_loaded, raw_feature_cols):
    """
    Ambil output load_jamshield() (Bagian 1) -> siapkan untuk dipakai
    fungsi-fungsi pcmef_fixed.py (rename kolom + override pf.FEATURES).

    HARUS dipanggil SEKALI di awal sebelum run_rf_jamshield /
    run_lstm_jamshield dipakai.
    """
    df = df_loaded.rename(columns={
        'pseudo_episode_id': 'pseudo_run_id',
        'time_sec': 'Time',
        'y': 'label',
    }).copy()
    pf.FEATURES = list(raw_feature_cols)   # override GLOBAL di modul pf
    return df


# ==========================================================================
# BAGIAN 3a -- RF-Stat (modified: 37-dim x 7 statistik dari window 15s)
# ==========================================================================

def run_rf_jamshield(df, seed, n_features, window_sec=15, step_sec=1):
    """
    Satu seed RF-Stat untuk JamShield. Analog run_rf_cell(..., arch='modified',
    protocol='pcmef') di RF Jamming, tapi num_classes=2 (bukan 3) dan
    FEATURES sudah di-override ke 37 kolom JamShield lewat prepare_jamshield_df().

    n_features: panjang RAW_FEATURE_COLS -- dipakai HANYA untuk sanity-check
    dimensi output, bukan untuk komputasi.
    """
    tr_df, va_df, te_df = pf.split_by_episode(df, seed=seed)
    Xtr_l, ytr, _ = pf.create_windows_keyed(tr_df, window_sec, step_sec)
    Xte_l, yte, _ = pf.create_windows_keyed(te_df, window_sec, step_sec)
    if not Xtr_l or not Xte_l or len(np.unique(ytr)) < 2:
        return None

    Xtr = pf.extract_tabular_features(Xtr_l)
    Xte = pf.extract_tabular_features(Xte_l)
    assert Xtr.shape[1] == n_features * 7, (
        f"Dimensi fitur tidak sesuai: {Xtr.shape[1]} != {n_features}*7. "
        f"Cek apakah pf.FEATURES sudah di-override dengan benar.")

    sc = StandardScaler().fit(Xtr)
    rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                                min_samples_split=5, min_samples_leaf=2,
                                n_jobs=-1, random_state=seed,
                                class_weight='balanced')
    rf.fit(sc.transform(Xtr), ytr)
    m = pf.compute_metrics(yte, rf.predict(sc.transform(Xte)), num_classes=2)
    m.update(model='RF_Stat', dataset='JamShield', protocol='pcmef',
             arch='modified', seed=int(seed),
             n_train=int(len(Xtr)), n_test=int(len(Xte)))
    return m


# ==========================================================================
# BAGIAN 3b -- LSTM (modified: 2-layer BiLSTM 128->64, SAMA seperti sel D)
# ==========================================================================

def run_lstm_jamshield(df, seed, n_features, window_sec=15, step_sec=1,
                       epochs=100, batch_size=32):
    """
    Satu seed LSTM untuk JamShield. Analog run_lstm_cell(..., arch='modified',
    protocol='pcmef') di RF Jamming -- arsitektur build_lstm('modified', ...)
    dipakai APA ADANYA (tidak diubah), cuma input_shape-nya otomatis
    menyesuaikan 37 fitur JamShield (bukan 5 fitur RF Jamming), dan
    num_classes=2 (bukan 3).
    """
    import tensorflow as tf
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    tr_df, va_df, te_df = pf.split_by_episode(df, seed=seed)
    Xtr_l, ytr, _ = pf.create_windows_keyed(tr_df, window_sec, step_sec)
    Xva_l, yva, _ = pf.create_windows_keyed(va_df, window_sec, step_sec)
    Xte_l, yte, _ = pf.create_windows_keyed(te_df, window_sec, step_sec)
    if min(len(Xtr_l), len(Xva_l), len(Xte_l)) == 0:
        return None

    max_len = max(max(len(x) for x in Xtr_l),
                  max(len(x) for x in Xva_l),
                  max(len(x) for x in Xte_l))
    Xtr, _ = pf.pad_list(Xtr_l, max_len)
    Xva, _ = pf.pad_list(Xva_l, max_len)
    Xte, _ = pf.pad_list(Xte_l, max_len)

    sc = StandardScaler().fit(Xtr.reshape(-1, n_features))
    rs = lambda A: sc.transform(A.reshape(-1, n_features)).reshape(A.shape)
    Xtr, Xva, Xte = rs(Xtr), rs(Xva), rs(Xte)

    model = pf.build_lstm('modified', (max_len, n_features), num_classes=2, seed=seed)
    model.compile(optimizer=Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs,
              batch_size=batch_size, verbose=0,
              callbacks=[EarlyStopping('val_loss', patience=15,
                                       restore_best_weights=True),
                         ReduceLROnPlateau('val_loss', factor=0.5,
                                           patience=5, min_lr=1e-6)])
    y_pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
    tf.keras.backend.clear_session()

    m = pf.compute_metrics(yte, y_pred, num_classes=2)
    m.update(model='LSTM', dataset='JamShield', protocol='pcmef',
             arch='modified', seed=int(seed),
             n_train=int(len(Xtr)), n_test=int(len(Xte)), max_len=int(max_len))
    return m


# ==========================================================================
# RUNNER RESUMABLE (satu file JSON per model x seed, aman kalau Colab putus)
# ==========================================================================

def run_jamshield_experiments(df, n_features, out_dir, seeds=None,
                              models=('RF_Stat', 'LSTM'), **kw):
    """
    Resumable runner: one JSON per model x seed. A cached file is reused only if
    pf.cache_is_valid() accepts its per-model code hash.
    """
    seeds = seeds or pf.SEEDS   # 20 seed yang sama dengan RF Jamming
    os.makedirs(out_dir, exist_ok=True)
    runner_map = {'RF_Stat': run_rf_jamshield, 'LSTM': run_lstm_jamshield}

    all_results = {m: [] for m in models}
    for model_name in models:
        runner = runner_map[model_name]
        model_kw = kw.get(model_name, {})
        for seed in seeds:
            fp = os.path.join(out_dir, f'JamShield_{model_name}_seed{seed}.json')
            if os.path.exists(fp):
                cached = json.load(open(fp))
                if pf.cache_is_valid(cached, model_name):
                    all_results[model_name].append(cached)
                    print(f'  [{model_name} seed {seed:4d}] sudah ada (hash cocok), dipakai ulang')
                    continue
                else:
                    print(f'  [{model_name} seed {seed:4d}] hash lama beda -> dihitung ulang')
            r = runner(df, seed, n_features, **model_kw)
            if r is None:
                print(f'  [{model_name} seed {seed}] dilewati (data tidak cukup)')
                continue
            r['code_hash'] = pf.model_code_hash(model_name)
            json.dump(r, open(fp, 'w'), indent=2)
            all_results[model_name].append(r)
            print(f'  [{model_name} seed {seed:4d}] acc={r["accuracy"]:.4f} '
                  f'f1={r["f1_macro"]:.4f}')
    return all_results


# ==========================================================================
# BAGIAN 4 -- AGREGASI STATISTIK MULTI-SEED
# ==========================================================================

def summarize_jamshield(results, metric='accuracy'):
    """Mean, SD, dan 95% bootstrap CI dari satu metrik lintas seed."""
    vals = np.array([r[metric] for r in results], dtype=float)
    if len(vals) == 0:
        return None
    lo, hi = pf.bootstrap_ci(vals)   # bootstrap CI dari mean vals itu sendiri
    return dict(model=results[0]['model'], metric=metric, n_seeds=len(vals),
               mean=float(vals.mean()),
               sd=float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
               ci95_lo=lo, ci95_hi=hi)


def print_jamshield_summary(all_results):
    print(f"{'Model':<10} {'Metrik':<12} {'n':>3} {'Mean':>8} {'SD':>8} "
          f"{'95% CI':>18}")
    print('-' * 65)
    for model_name, results in all_results.items():
        if not results:
            print(f'{model_name:<10} -- tidak ada hasil (semua dilewati) --')
            continue
        for metric in ('accuracy', 'f1_macro'):
            s = summarize_jamshield(results, metric)
            print(f"{s['model']:<10} {s['metric']:<12} {s['n_seeds']:>3} "
                  f"{s['mean']*100:>7.2f}% {s['sd']*100:>7.2f}% "
                  f"[{s['ci95_lo']*100:>6.2f}%, {s['ci95_hi']*100:>6.2f}%]")


In [ ]:
N_FEATURES = len(RAW_FEATURE_COLS)
print("Jumlah fitur:", N_FEATURES)

df = prepare_jamshield_df(df_loaded, RAW_FEATURE_COLS)
print("Kolom siap:", [c for c in ['pseudo_run_id', 'Time', 'label'] if c in df.columns])

In [ ]:
df, feasible_ids, n_dropped = pf.filter_feasible_episodes(df, min_duration_sec=15)
print(f"Episode dibuang (durasi < 15s): {n_dropped}")

### Run RF-Stat: 20 seeds

In [ ]:
OUT_DIR = '/content/drive/MyDrive/JamShield_Dataset/results'

results = run_jamshield_experiments(df, N_FEATURES, OUT_DIR, models=('RF_Stat',))

### Run LSTM: 20 seeds

Use a GPU runtime (T4). Resumable, like RF-Stat.

In [ ]:
results_lstm = run_jamshield_experiments(df, N_FEATURES, OUT_DIR, models=('LSTM',))
results.update(results_lstm)

### Summary statistics (mean ± SD, 95% bootstrap CI, 20 seeds)

In [ ]:
print_jamshield_summary(results)

### Save final summary to Drive

In [ ]:
import json

summary_export = {}
for model_name, res_list in results.items():
    if not res_list:
        continue
    summary_export[model_name] = {
        'accuracy': summarize_jamshield(res_list, 'accuracy'),
        'f1_macro': summarize_jamshield(res_list, 'f1_macro'),
        'n_seeds_completed': len(res_list),
    }

SUMMARY_PATH = '/content/drive/MyDrive/JamShield_Dataset/jamshield_summary_20seed.json'
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary_export, f, indent=2)
print(f"Tersimpan: {SUMMARY_PATH}")
print(json.dumps(summary_export, indent=2))

In [ ]:
"""
Audit A5: why is RF-Stat accuracy on JamShield so high (99.29% +- 0.63%, 20 seeds)?
Is it easy separability, or leakage / duplication between train and test?

  A5a. Exact duplicate raw rows (before windowing) between train and test
  A5b. Nearest-neighbour distance test->train vs train->train, in the same
       feature space RF-Stat uses (259-dim: 37 features x 7 statistics,
       standardised with a train-fitted scaler)
  A5c. source_file overlap between train and test (JamShield-specific): share of
       test episodes whose source file also appears in train. pseudo_episode_id =
       (source_file, station, 60 s bucket), and the split groups by episode, not
       by file, so episodes from the same file (same jamming type and conditions)
       can fall in both train and test. This is by design, not a bug, but it is
       context for the high accuracy.
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


# ==========================================================================
# A5a -- duplikasi baris mentah persis
# ==========================================================================

def audit_raw_duplicates(tr_df, te_df, raw_feature_cols):
    """
    Cek apakah ada baris MENTAH (sebelum windowing) di test yang PERSIS SAMA
    dengan baris di train, berdasarkan seluruh nilai RAW_FEATURE_COLS.
    """
    tr_keys = set(map(tuple, tr_df[raw_feature_cols].values))
    te_vals = list(map(tuple, te_df[raw_feature_cols].values))
    n_exact = sum(1 for v in te_vals if v in tr_keys)
    pct = 100.0 * n_exact / len(te_vals) if te_vals else 0.0
    return dict(n_test_rows=len(te_vals), n_exact_match_in_train=n_exact,
               pct_exact_match=pct)


# ==========================================================================
# A5b -- jarak NN test->train vs train->train, di ruang fitur RF-Stat
# ==========================================================================

def audit_nn_distance(pf, tr_df, te_df, window_sec=15, step_sec=1, seed=42):
    """
    Bangun window + 259-dim tabular features PERSIS seperti run_rf_jamshield(),
    lalu bandingkan jarak Euclidean (di ruang terstandarisasi):
      - test  -> tetangga terdekat di train
      - train -> tetangga terdekat di train LAIN (leave-one-out, exclude diri sendiri)

    Kalau jarak test->train jauh LEBIH KECIL daripada train->train, itu tanda
    test "terlalu mirip" train -- indikasi kebocoran, bukan model yang bagus.
    """
    Xtr_l, ytr, ktr = pf.create_windows_keyed(tr_df, window_sec, step_sec)
    Xte_l, yte, kte = pf.create_windows_keyed(te_df, window_sec, step_sec)
    if not Xtr_l or not Xte_l:
        return None

    Xtr = pf.extract_tabular_features(Xtr_l)
    Xte = pf.extract_tabular_features(Xte_l)

    sc = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr)
    Xte_s = sc.transform(Xte)

    # test -> train
    nn_tr = NearestNeighbors(n_neighbors=1).fit(Xtr_s)
    d_te_tr, _ = nn_tr.kneighbors(Xte_s)
    d_te_tr = d_te_tr.ravel()

    # train -> train (leave-one-out: ambil k=2 lalu buang jarak ke diri sendiri, yaitu 0)
    nn_tr2 = NearestNeighbors(n_neighbors=2).fit(Xtr_s)
    d_tr_tr, _ = nn_tr2.kneighbors(Xtr_s)
    d_tr_tr = d_tr_tr[:, 1]   # kolom 0 = jarak ke diri sendiri (0), kolom 1 = tetangga asli

    p05_train = float(np.percentile(d_tr_tr, 5))
    pct_test_closer_than_p05 = float(100.0 * np.mean(d_te_tr < p05_train))

    return dict(
        n_train_windows=len(Xtr), n_test_windows=len(Xte),
        d_test_to_train_mean=float(d_te_tr.mean()),
        d_test_to_train_median=float(np.median(d_te_tr)),
        d_test_to_train_min=float(d_te_tr.min()),
        d_train_to_train_mean=float(d_tr_tr.mean()),
        d_train_to_train_median=float(np.median(d_tr_tr)),
        p05_train_to_train=p05_train,
        pct_test_closer_than_p05_train=pct_test_closer_than_p05,
    )


# ==========================================================================
# A5c -- overlap source_file antara train & test (BARU, spesifik JamShield)
# ==========================================================================

def audit_file_overlap(tr_df, te_df):
    """
    Episode di JamShield = (source_file, station, bucket). Cek berapa persen
    EPISODE test yang source_file-nya JUGA punya episode di train (beda
    station/menit, tapi kondisi jamming identik).

    Ini BUKAN bug (episode-level split memang didesain begini, sama seperti
    RF Jamming yang episode-nya juga bisa berasal dari Scenario yang sama),
    tapi PENTING dilaporkan sebagai konteks kenapa akurasi bisa setinggi ini.
    """
    tr_files = set(tr_df['source_file'].unique())
    te_ep_files = te_df.groupby('pseudo_run_id')['source_file'].first()
    n_overlap = te_ep_files.isin(tr_files).sum()
    pct = 100.0 * n_overlap / len(te_ep_files) if len(te_ep_files) else 0.0
    return dict(n_test_episodes=len(te_ep_files),
               n_test_episodes_file_also_in_train=int(n_overlap),
               pct_test_episodes_file_also_in_train=pct,
               n_unique_files_train=len(tr_files),
               n_unique_files_test=te_ep_files.nunique())


# ==========================================================================
# RUNNER -- jalankan ketiganya untuk satu seed
# ==========================================================================

def run_audit_a5_jamshield(pf, df, raw_feature_cols, seed=42, window_sec=15, step_sec=1):
    tr_df, va_df, te_df = pf.split_by_episode(df, seed=seed)

    print(f"=== AUDIT A5-STYLE JAMSHIELD (seed {seed}) ===\n")

    print("--- A5a: duplikasi baris mentah persis (test vs train) ---")
    r_dup = audit_raw_duplicates(tr_df, te_df, raw_feature_cols)
    for k, v in r_dup.items():
        print(f"  {k:35s}: {v}")
    if r_dup['pct_exact_match'] > 1.0:
        print("  >> PERHATIAN: >1% baris test identik persis dengan baris train.")
    else:
        print("  >> OK, tidak ada duplikasi baris mentah yang signifikan.")

    print("\n--- A5b: jarak nearest-neighbor (ruang fitur RF-Stat, 259-dim) ---")
    r_nn = audit_nn_distance(pf, tr_df, te_df, window_sec, step_sec, seed)
    for k, v in r_nn.items():
        print(f"  {k:35s}: {v:.4f}" if isinstance(v, float) else f"  {k:35s}: {v}")
    if r_nn['pct_test_closer_than_p05_train'] > 20.0:
        print("  >> PERHATIAN: >20% window test lebih dekat ke train daripada "
              "train ke sesamanya sendiri (p05). Indikasi window test 'terlalu "
              "mirip' train -- akurasi tinggi mungkin sebagian karena ini.")
    else:
        print("  >> OK, window test TIDAK sistematis lebih dekat ke train "
              "dibanding train ke sesamanya sendiri.")

    print("\n--- A5c: overlap source_file antara train & test (spesifik JamShield) ---")
    r_file = audit_file_overlap(tr_df, te_df)
    for k, v in r_file.items():
        print(f"  {k:35s}: {v:.2f}" if isinstance(v, float) else f"  {k:35s}: {v}")
    pct_ov = r_file['pct_test_episodes_file_also_in_train']
    print(f"  >> Context (not a bug): {pct_ov:.1f}% of test episodes come from files whose "
          f"jamming conditions also appear in train (different station/minute only). "
          f"Expected for an episode-level split; the model partly learns from identical "
          f"jamming conditions rather than generalising to unseen ones.")

    return dict(seed=seed, raw_duplicates=r_dup, nn_distance=r_nn, file_overlap=r_file)


def run_audit_a5_multi_seed(pf, df, raw_feature_cols, seeds=None, window_sec=15, step_sec=1):
    """Jalankan audit di beberapa seed sekaligus, ringkas hasilnya."""
    seeds = seeds or pf.SEEDS[:5]   # default 5 seed cukup untuk audit (tidak perlu 20)
    all_r = []
    for seed in seeds:
        tr_df, va_df, te_df = pf.split_by_episode(df, seed=seed)
        r_dup = audit_raw_duplicates(tr_df, te_df, raw_feature_cols)
        r_nn = audit_nn_distance(pf, tr_df, te_df, window_sec, step_sec, seed)
        r_file = audit_file_overlap(tr_df, te_df)
        all_r.append(dict(seed=seed, raw_duplicates=r_dup, nn_distance=r_nn, file_overlap=r_file))
        print(f"[seed {seed:4d}] pct_exact_dup={r_dup['pct_exact_match']:.2f}%  "
              f"pct_test<p05_train={r_nn['pct_test_closer_than_p05_train']:.1f}%  "
              f"pct_file_overlap={r_file['pct_test_episodes_file_also_in_train']:.1f}%")

    pct_dup = np.mean([r['raw_duplicates']['pct_exact_match'] for r in all_r])
    pct_close = np.mean([r['nn_distance']['pct_test_closer_than_p05_train'] for r in all_r])
    pct_overlap = np.mean([r['file_overlap']['pct_test_episodes_file_also_in_train'] for r in all_r])
    print(f"\n=== RATA-RATA {len(seeds)} SEED ===")
    print(f"  Duplikasi baris mentah        : {pct_dup:.2f}%")
    print(f"  Test < p05(train->train)      : {pct_close:.1f}%")
    print(f"  Overlap source_file train/test: {pct_overlap:.1f}%")
    return all_r

In [ ]:
hasil_audit = run_audit_a5_jamshield(pf, df, RAW_FEATURE_COLS, seed=42)

## Episodes per split

In [ ]:
tr_df, va_df, te_df = pf.split_by_episode(df, seed=42)

def ep_stats(d, name):
    eps = d['pseudo_run_id'].nunique()
    per_class = d.groupby('pseudo_run_id')['label'].first().value_counts().to_dict()
    print(f"{name:6s}: rows={len(d):6d}  episodes={eps:4d}  per_class={per_class}")

ep_stats(tr_df, "TRAIN")
ep_stats(va_df, "VAL")
ep_stats(te_df, "TEST")

In [ ]:
import numpy as np
import pandas as pd


def check_split_stability(pf, df, seeds=None):
    """
    Jalankan split_by_episode() untuk setiap seed, catat jumlah episode
    dan jumlah baris tiap partisi (train/val/test), lalu ringkas variasinya.
    """
    seeds = seeds or pf.SEEDS
    rows = []
    for seed in seeds:
        tr_df, va_df, te_df = pf.split_by_episode(df, seed=seed)
        rows.append(dict(
            seed=seed,
            train_eps=tr_df['pseudo_run_id'].nunique(),
            val_eps=va_df['pseudo_run_id'].nunique(),
            test_eps=te_df['pseudo_run_id'].nunique(),
            train_rows=len(tr_df), val_rows=len(va_df), test_rows=len(te_df),
        ))

    t = pd.DataFrame(rows)
    print(t.to_string(index=False))
    print()
    print("=== Ringkasan konsistensi across 20 seed ===")
    for col in ['train_eps', 'val_eps', 'test_eps']:
        print(f"{col:10s}: mean={t[col].mean():.2f}  std={t[col].std():.4f}  "
              f"min={t[col].min()}  max={t[col].max()}")

    total_eps_per_seed = (t['train_eps'] + t['val_eps'] + t['test_eps']).unique()
    print(f"\nTotal episode per seed (harus selalu sama): {total_eps_per_seed}")

    all_same = (t['train_eps'].nunique() == 1 and
                t['val_eps'].nunique() == 1 and
                t['test_eps'].nunique() == 1)
    if all_same:
        print("\n>> VALID: jumlah episode train/val/test IDENTIK di semua "
              f"{len(seeds)} seed -- proporsi 60/20/20 (per kelas) terjaga "
              "persis sama, tidak ada seed yang menyimpang. Variasi antar "
              "seed hanya pada BARIS (karena panjang tiap episode berbeda), "
              "bukan pada JUMLAH episode-nya.")
    else:
        print("\n>> PERHATIAN: jumlah episode BERBEDA antar seed -- cek "
              "apakah ada masalah pada df atau split_by_episode().")

    return t

## Split validation

In [ ]:
t = check_split_stability(pf, df)

In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/Detector_Results/pcmef_v2_infra.py', '/content/pcmef_v2_infra.py')
import pcmef_v2_infra as v2

print("=== Sanity check JamShield ===")
v2.run_sanity_suite(pf, df, seeds=(42,))